### 4/5/2026 - SLIMs analysis notebook, running an enrichment analysis on SLIMs identified previous by slimprob. 

approach:
1. Load csv of viral tiles annotated with SLIMs
2. Count frequency of each and every slim in background set of viral tiles
3. for each subdivision of peptides, 
4. examine proportion of each SLIM in this population of peptides
5. for each slim, create contingency table
6. Determine log odds and p-value with fischer's exact test
7. Perform FDR correction at 0.1 ~ q value 

In [ ]:
#load modules 
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 
import scipy.stats

from matplotlib import font_manager

from matplotlib import rcParams
rcParams['font.sans-serif'] = "Arial"
rcParams['font.family'] = "Arial"



plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial'
#load our big df
occ = pd.read_csv("SLIMs/Data/all_peptides_slimprob_summary.occ.split.csv",header=0)


occ["Seq"].nunique()
#ONLY VIRAL TILES:
occ = occ[occ['AccNum'].str.contains('VT')]


occ["peptide_id"] = occ["Seq"].str.split("__").str[0]
#load all my prev data:
input_df = pd.read_csv("VTs_withprops_SLiMs.csv",header=0)
len(input_df)

input_df = input_df[input_df['identifier'].str.contains('VT')]



In [ ]:

#now we label with the properties
occ_labelled = occ.merge(input_df[["identifier", "Disorder_perc", "Sig_RITA", "FC_RITA"]], left_on="AccNum", right_on="identifier", how="left")


In [ ]:
def test_effect(row):
    
    if row['Sig_RITA'] == 'No':
        return 'None' 
    elif row['FC_RITA'] > 0:
        return 'Protective'
    else:
        return 'Deleterious'
    
occ_labelled['Effect'] = occ_labelled.apply(lambda row: test_effect(row), axis=1)

In [ ]:
#slice by effect protective and deleterious


summary_text = ""
Protective_occ_labelled = occ_labelled[occ_labelled['Effect']=='Protective']

Deleterious_occ_labelled = occ_labelled[occ_labelled['Effect']=='Deleterious']

print(f"Total of {occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (background) with total {len(occ_labelled)} SLIMs")
print(f"Total of {Protective_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Enriched / Protective) with total {len(Protective_occ_labelled)} SLIMs")
print(f"Total of {Deleterious_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Deleterious / Depleted) with total {len(Deleterious_occ_labelled)} SLIMs")
summary_text+=f"Total of {occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (background) with total {len(occ_labelled)} SLIMs"
summary_text+=f"Total of {Protective_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Enriched / Protective) with total {len(Protective_occ_labelled)} SLIMs"
summary_text+=f"Total of {Deleterious_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Deleterious / Depleted) with total {len(Deleterious_occ_labelled)} SLIMs"


In [ ]:
results_all = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])
results_per = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])

background_motif_counts = occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_counts = Protective_occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_list = protective_motif_counts.index

deleterious_motif_counts = Deleterious_occ_labelled.groupby("Motif")['identifier'].nunique()

deleterious_motif_list = deleterious_motif_counts.index

#check if any have zero in background
zero_bg_motifs_protective = [m for m in protective_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_protective)

zero_bg_motifs_deleterious = [m for m in deleterious_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_deleterious)



for (subset, name) in [(Protective_occ_labelled, "Protective"), (Deleterious_occ_labelled, "Deleterious")]:
    print(f"Analyzing subset of {name} peptides")
    
    for motif in subset['Motif'].unique():

        # Enriched set counts
        subset_filtered = subset[subset['Motif'] == motif]
        #count every time this motif appears in a peptide once
        E_WM_per = subset_filtered['identifier'].nunique()
        E_WOM_per = subset['identifier'].nunique() - E_WM_per
        
        #or, count total
        E_WM = len(subset_filtered)
        E_WOM = len(subset) - E_WM

        # Non-enriched set counts
        background_filtered = occ_labelled[occ_labelled['Motif'] == motif]
        #following line looks for unique count of motifs
        NE_WM_per = background_filtered['identifier'].nunique()
        NE_WOM_per = occ_labelled['identifier'].nunique() - NE_WM_per
        
        #following line couns total
        NE_WM = len(background_filtered)
        NE_WOM = len(occ_labelled) - NE_WM
        
         


        # Skip untestable hypotheses explicitly
        if E_WM == 0:
            continue
        
        if E_WM_per == 0:
            continue

        contingency_table = np.array([
            [E_WM,  NE_WM],
            [E_WOM, NE_WOM]
        ])
        
        contingency_table_per = np.array([
            [E_WM_per,  NE_WM_per],
            [E_WOM_per, NE_WOM_per]
        ])

        oddsratio, pvalue = scipy.stats.fisher_exact(
            contingency_table,
            alternative='greater'
        )
        oddsratio_per, pvalue_per = scipy.stats.fisher_exact(
            contingency_table_per,
            alternative='greater'
        )



        results_all.loc[len(results_all)] = [motif, contingency_table, oddsratio, pvalue, name]
        results_per.loc[len(results_all)] = [motif, contingency_table_per, oddsratio_per, pvalue_per, name]




In [ ]:
#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals
correction_level = 0.1

results_all_filtered = results_all[results_all['P-value'] < 1.0]

results_protective = results_all_filtered[results_all_filtered['Subset']=='Protective']

results_deleterious = results_all_filtered[results_all_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=correction_level, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=correction_level, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious

summary_text+=f"\n\nFor BULK-SLIM-ANALYSIS;  Quantity of Enriched SLIMs in the Protective Subset, Examining Exact SLIMs:\n{len(results_protective[results_protective['significant']==True])}"
summary_text+=f"\nThey are as follows:\n {results_protective[results_protective['significant']==True].to_string()}"
summary_text+=f"\n\nFor BULK-SLIM-ANALYSIS;  Quantity of Enriched SLIMs in the Deleterious Subset, Examining Exact SLIMs:\n{len(results_deleterious[results_deleterious['significant']==True])}"
summary_text+=f"\nThey are as follows:\n {results_deleterious[results_deleterious['significant']==True].to_string()}"

results_protective.to_csv("SLIMs/Results/Plots/protective_filter_for_specific_SLIMS.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/deleterious_filter_for_specific_SLIMS.csv",header=True,index=None)



In [ ]:

custom_palette = {True: "#EF345C", False: "gray"}

#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=6)
plt.axhline(correction_level)
plt.title(f"results for protective subset considering enrichment \nin bulk quantites of peptides with this slim (correction level = {correction_level})")
sns.despine(offset=10, trim=True)
plt.gca().legend_.remove() # Removes the legend from current axes
plt.tight_layout()
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/protective_filter_for_specific_SLIMS.pdf",dpi=300)


plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=6)
plt.axhline(correction_level)

#plt.title(f"Results for deleterious subset considering enrichment \nin bulk quantities of unique peptides with this slim (correction level) = {correction_level}")
plt.xlim(0, 12)

sns.despine(offset=10, trim=False)
plt.gca().legend_.remove() # Removes the legend from current axes
plt.tight_layout()
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/deleterious_filter_for_specific_SLIMS.pdf",dpi=300)

plt.show()




In [ ]:
#ALL PER 

#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals

results_per_filtered = results_per[(results_per['P-value'] < 1.0)]

results_protective = results_per_filtered[results_per_filtered['Subset']=='Protective']

results_deleterious = results_per_filtered[results_per_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=0.10, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=0.10, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious

summary_text+=f"\n\nFor PER-SLIM-ANALYSIS; Quantity of Enriched SLIMs in the Protective Subset, Examining Exact SLIMs:\n{len(results_protective[results_protective['significant']==True])}"
summary_text+=f"\nThey are as follows:\n {results_protective[results_protective['significant']==True].to_string()}"
summary_text+=f"\n\nFor PER-SLIM-ANALYSIS; Quantity of Enriched SLIMs in the Deleterious Subset, Examining Exact SLIMs:\n{len(results_deleterious[results_deleterious['significant']==True])}"
summary_text+=f"\nThey are as follows:\n {results_deleterious[results_deleterious['significant']==True].to_string()}"

#save


results_protective.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_filter_for_specific_SLIMS.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_filter_for_specific_SLIMS.csv",header=True,index=None)


custom_palette = {True: "#EF345C", False: "gray"}

#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=6)
plt.axhline(correction_level)
#plt.title(f"results for protective subset considering enrichment \nin amount of unique peptides with this slim (correction level = {correction_level})")
sns.despine(offset=10, trim=True)
plt.gca().legend_.remove() # Removes the legend from current axes

plt.tight_layout()
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_filter_for_specific_SLIMS.pdf",dpi=300)


plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=6)
plt.axhline(correction_level)

#plt.title(f"Results for deleterious subset considering enrichment \nin amount of unique peptides with this slim (correction level) = {correction_level}")
plt.xlim(0, 12)

sns.despine(offset=10, trim=False)
plt.gca().legend_.remove() # Removes the legend from current axes
plt.tight_layout()
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_filter_for_specific_SLIMS.pdf",dpi=300)

plt.show()

In [ ]:
#retry with broader categories
results_all = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])
results_per = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])

def sort_broad_category(value):
    first_second = value.split('_')[0]+"_"+value.split('_')[1]
    return first_second

occ_labelled["Motif"] = occ_labelled['Motif'].apply(lambda x: sort_broad_category(x))

#slice by effect protective and deleterious

Protective_occ_labelled = occ_labelled[occ_labelled['Effect']=='Protective']

Deleterious_occ_labelled = occ_labelled[occ_labelled['Effect']=='Deleterious']

print(f"Total of {occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (background) with total {len(occ_labelled)} SLIMs")
print(f"Total of {Protective_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Enriched / Protective) with total {len(Protective_occ_labelled)} SLIMs")
print(f"Total of {Deleterious_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Deleterious / Depleted) with total {len(Deleterious_occ_labelled)} SLIMs")


background_motif_counts = occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_counts = Protective_occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_list = protective_motif_counts.index

deleterious_motif_counts = Deleterious_occ_labelled.groupby("Motif")['identifier'].nunique()

deleterious_motif_list = deleterious_motif_counts.index

#check if any have zero in background
zero_bg_motifs_protective = [m for m in protective_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_protective)

zero_bg_motifs_deleterious = [m for m in deleterious_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_deleterious)



for (subset, name) in [(Protective_occ_labelled, "Protective"), (Deleterious_occ_labelled, "Deleterious")]:
    print(f"Analyzing subset of {name} peptides")
    
    for motif in subset['Motif'].unique():

        # Enriched set counts
        subset_filtered = subset[subset['Motif'] == motif]
        #count every time this motif appears in a peptide once
        E_WM_per = subset_filtered['identifier'].nunique()
        E_WOM_per = subset['identifier'].nunique() - E_WM_per
        
        #or, count total
        E_WM = len(subset_filtered)
        E_WOM = len(subset) - E_WM

        # Non-enriched set counts
        background_filtered = occ_labelled[occ_labelled['Motif'] == motif]
        #following line looks for unique count of motifs
        NE_WM_per = background_filtered['identifier'].nunique()
        NE_WOM_per = occ_labelled['identifier'].nunique() - NE_WM_per
        
        #following line couns total
        NE_WM = len(background_filtered)
        NE_WOM = len(occ_labelled) - NE_WM
        
         


        # Skip untestable hypotheses explicitly
        if E_WM == 0:
            continue
        
        if E_WM_per == 0:
            continue

        contingency_table = np.array([
            [E_WM,  NE_WM],
            [E_WOM, NE_WOM]
        ])
        
        contingency_table_per = np.array([
            [E_WM_per,  NE_WM_per],
            [E_WOM_per, NE_WOM_per]
        ])

        oddsratio, pvalue = scipy.stats.fisher_exact(
            contingency_table,
            alternative='greater'
        )
        oddsratio_per, pvalue_per = scipy.stats.fisher_exact(
            contingency_table_per,
            alternative='greater'
        )



        results_all.loc[len(results_all)] = [motif, contingency_table, oddsratio, pvalue, name]
        results_per.loc[len(results_all)] = [motif, contingency_table_per, oddsratio_per, pvalue_per, name]




In [ ]:
#ALL ALL 
#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals
correction_level = 0.1

results_all_filtered = results_all[results_all['P-value'] < 1.0]

results_protective = results_all_filtered[results_all_filtered['Subset']=='Protective']

results_deleterious = results_all_filtered[results_all_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=correction_level, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=correction_level, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=6)
plt.axhline(correction_level)
#plt.title(f"results for protective subset considering enrichment \nin bulk, secondary level of filtering with this slim (correction level = {correction_level})")


sns.despine(offset=10, trim=False)
plt.tight_layout()

#print all 

print(f"Enriched Secondary Categories of SLIMS in Protective peptides:\n\n{results_protective[results_protective['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/protective_secondary_broadness_SLIMS.pdf",dpi=300)
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=6)
plt.axhline(correction_level)


#plt.title(f"Results for deleterious subset considering enrichment \nin bulk, secondary level of filtering with this slim (correction level) = {correction_level}")
sns.despine(offset=10, trim=False)
plt.tight_layout()
print(f"Enriched Secondary Categories of SLIMS in Deleterious peptides:\n\n{results_deleterious[results_deleterious['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/deleterious_secondary_broadness_SLIMS.pdf",dpi=300)

plt.show()

results_protective.to_csv("SLIMs/Results/Plots/Bulk_Enrichment/protective_secondary_broadness_SLIMS.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/Bulk_Enrichment/deleterious_secondary_broadness_SLIMS.csv",header=True,index=None)

sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant")
plt.axhline(correction_level)
plt.title(f"Enrichment in protective peptides considering the enrichment \nin absolute quantity of motif (correction level = {correction_level})")


for i, row in results_protective.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot
        textcoords='offset points'
    )
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant')
plt.axhline(correction_level)
plt.title(f"Results for deleterious subset considering the enrichment  \nin absolute quantity of motif (correction level = {correction_level})")
for i, row in results_deleterious.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot
        textcoords='offset points'
    )
plt.show()




#ALL PER 

#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals

results_per_filtered = results_per[(results_per['P-value'] < 1.0)]

results_protective = results_per_filtered[results_per_filtered['Subset']=='Protective']

results_deleterious = results_per_filtered[results_per_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=0.10, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=0.10, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


custom_palette = {True: "#EF345C", False: "gray"}

#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=6)
plt.axhline(correction_level)
#plt.title(f"results for protective subset considering enrichment \nin amount of unique peptides with this slim (correction level = {correction_level})")


sns.despine(offset=10, trim=False)

#print all 
plt.tight_layout()
print(f"Enriched Secondary Categories of SLIMS in Protective peptides:\n\n{results_protective[results_protective['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_secondary_broadness_SLIMS.pdf",dpi=300)
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=6)
plt.axhline(correction_level)


plt.title(f"Results for deleterious subset considering enrichment \nin amount of unique peptides with this slim (correction level) = {correction_level}")
sns.despine(offset=10, trim=False)
print(f"Enriched Secondary Categories of SLIMS in Deleterious peptides:\n\n{results_deleterious[results_deleterious['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.tight_layout()
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_secondary_broadness_SLIMS.pdf",dpi=300)




plt.show()

#save



results_protective.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_secondary_broadness_SLIMS.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_secondary_broadness_SLIMS.csv",header=True,index=None)


In [ ]:
results_all = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])
results_per = pd.DataFrame(columns=['SLIM Motif', 'Contingency Table', "Odds Ratio", "P-value", "Subset"])

def sort_broad_category(value):
    first = value.split('_')[0]
    return first

occ_labelled["Motif"] = occ_labelled['Motif'].apply(lambda x: sort_broad_category(x))

#slice by effect protective and deleterious

Protective_occ_labelled = occ_labelled[occ_labelled['Effect']=='Protective']

Deleterious_occ_labelled = occ_labelled[occ_labelled['Effect']=='Deleterious']

print(f"Total of {occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (background) with total {len(occ_labelled)} SLIMs")
print(f"Total of {Protective_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Enriched / Protective) with total {len(Protective_occ_labelled)} SLIMs")
print(f"Total of {Deleterious_occ_labelled['AccNum'].nunique()} unique viral tiles under consideration (Deleterious / Depleted) with total {len(Deleterious_occ_labelled)} SLIMs")


background_motif_counts = occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_counts = Protective_occ_labelled.groupby("Motif")['identifier'].nunique()

protective_motif_list = protective_motif_counts.index

deleterious_motif_counts = Deleterious_occ_labelled.groupby("Motif")['identifier'].nunique()

deleterious_motif_list = deleterious_motif_counts.index

#check if any have zero in background
zero_bg_motifs_protective = [m for m in protective_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_protective)

zero_bg_motifs_deleterious = [m for m in deleterious_motif_list if background_motif_counts.get(m, 0) == 0]

print(zero_bg_motifs_deleterious)



for (subset, name) in [(Protective_occ_labelled, "Protective"), (Deleterious_occ_labelled, "Deleterious")]:
    print(f"Analyzing subset of {name} peptides")
    
    for motif in subset['Motif'].unique():

        # Enriched set counts
        subset_filtered = subset[subset['Motif'] == motif]
        #count every time this motif appears in a peptide once
        E_WM_per = subset_filtered['identifier'].nunique()
        E_WOM_per = subset['identifier'].nunique() - E_WM_per
        
        #or, count total
        E_WM = len(subset_filtered)
        E_WOM = len(subset) - E_WM

        # Non-enriched set counts
        background_filtered = occ_labelled[occ_labelled['Motif'] == motif]
        #following line looks for unique count of motifs
        NE_WM_per = background_filtered['identifier'].nunique()
        NE_WOM_per = occ_labelled['identifier'].nunique() - NE_WM_per
        
        #following line couns total
        NE_WM = len(background_filtered)
        NE_WOM = len(occ_labelled) - NE_WM
        
         


        # Skip untestable hypotheses explicitly
        if E_WM == 0:
            continue
        
        if E_WM_per == 0:
            continue

        contingency_table = np.array([
            [E_WM,  NE_WM],
            [E_WOM, NE_WOM]
        ])
        
        contingency_table_per = np.array([
            [E_WM_per,  NE_WM_per],
            [E_WOM_per, NE_WOM_per]
        ])

        oddsratio, pvalue = scipy.stats.fisher_exact(
            contingency_table,
            alternative='greater'
        )
        oddsratio_per, pvalue_per = scipy.stats.fisher_exact(
            contingency_table_per,
            alternative='greater'
        )



        results_all.loc[len(results_all)] = [motif, contingency_table, oddsratio, pvalue, name]
        results_per.loc[len(results_all)] = [motif, contingency_table_per, oddsratio_per, pvalue_per, name]




In [ ]:
#ALL ALL 
#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals
correction_level = 0.1

results_all_filtered = results_all[results_all['P-value'] < 1.0]

results_protective = results_all_filtered[results_all_filtered['Subset']=='Protective']

results_deleterious = results_all_filtered[results_all_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=correction_level, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=correction_level, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


results_protective.to_csv("SLIMs/Results/Plots/Bulk_Enrichment/protective_broadest_SLIM_categories.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/Bulk_Enrichment/deleterious_broadest_SLIM_categories.csv",header=True,index=None)

sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant")
plt.axhline(correction_level)
plt.title(f"Enrichment in protective peptides considering the enrichment \nin absolute quantity of motif (correction level = {correction_level})")

for i, row in results_protective.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot
        textcoords='offset points'
    )
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant')
plt.axhline(correction_level)
plt.title(f"Results for deleterious subset considering the enrichment  \nin absolute quantity of motif (correction level = {correction_level})")
for i, row in results_deleterious.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot
        textcoords='offset points'
    )
plt.show()


sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=8)
plt.axhline(correction_level)
plt.title(f"results for protective subset considering enrichment \nin bulk amount instances of this broadest slim category (correction level = {correction_level})")


sns.despine(offset=10, trim=False)

#print all 
plt.tight_layout()

print(f"Enriched Broadest Categories of SLIMS in Protective peptides:\n\n{results_protective[results_protective['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/protective_broadest_SLIM_categories.pdf",dpi=300)
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=8)
plt.axhline(correction_level)

plt.tight_layout()

plt.title(f"Results for deleterious subset considering enrichment \nin bulk amount instances of this broadest slim category (correction level) = {correction_level}")
sns.despine(offset=10, trim=False)
print(f"Enriched Broadest Categories of SLIMS in Deleterious peptides:\n\n{results_deleterious[results_deleterious['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Bulk_Enrichment/deleterious_broadest_SLIM_categories.pdf",dpi=300)

plt.show()

#ALL PER 

#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals

results_per_filtered = results_per[(results_per['P-value'] < 1.0)]

results_protective = results_per_filtered[results_per_filtered['Subset']=='Protective']

results_deleterious = results_per_filtered[results_per_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=0.10, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=0.10, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


custom_palette = {True: "#EF345C", False: "gray"}

#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette, s=8)
plt.axhline(correction_level)
plt.title(f"results for protective subset considering enrichment \nin amount of unique peptides with this slim (correction level = {correction_level})")

sns.despine(offset=10, trim=False)

#print all 
plt.tight_layout()
print(f"Enriched Broadest Categories of SLIMS in Protective peptides:\n\n{results_protective[results_protective['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_broadest_SLIM_categories.pdf",dpi=300)
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette, s=8)
plt.axhline(correction_level)


plt.title(f"Results for deleterious subset considering enrichment \nin amount of unique peptides with this slim (correction level) = {correction_level}")
sns.despine(offset=10, trim=False)
plt.tight_layout()

print(f"Enriched Broadest Categories of SLIMS in Deleterious peptides:\n\n{results_deleterious[results_deleterious['significant']==True]['SLIM Motif'].dropna().unique()}")
plt.savefig("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_broadest_SLIM_categories.pdf",dpi=300)

plt.show()
#save



results_protective.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/protective_broadest_SLIM_categories.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Plots/Per_Peptide_Enrichment/deleterious_broadest_SLIM_categories.csv",header=True,index=None)


In [ ]:
#ALL ALL 
#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals
correction_level = 0.1

results_all_filtered = results_all[results_all['P-value'] < 1.0]

results_protective = results_all_filtered[results_all_filtered['Subset']=='Protective']

results_deleterious = results_all_filtered[results_all_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=correction_level, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=correction_level, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


results_protective.to_csv("SLIMs/Results/Tables/Results_bulk_BROAD_Protective.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Tables/Results_bulk_BROAD_deleterious.csv",header=True,index=None)

sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", s=6)
plt.axhline(correction_level)
plt.title(f"Enrichment in protective peptides considering the enrichment \nin absolute quantity of motif (correction level = {correction_level})")

for i, row in results_protective.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot
        textcoords='offset points'
    )
plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', s=6)
plt.axhline(correction_level)
plt.title(f"Results for deleterious subset considering the enrichment  \nin absolute quantity of motif (correction level = {correction_level})")
for i, row in results_deleterious.iterrows():
    plt.annotate(
        text=str(row['SLIM Motif']),  # Ensure text is string
        xy=(float(row['Odds Ratio']), float(row['q-value'])), # Ensure coordinates are floats
        xytext=(5, 5), # Offset text to avoid covering dot 
        textcoords='offset points'
    )
plt.show()




#ALL PER 

#correction, q-values..
from statsmodels.stats.multitest import multipletests

#filter for non-valid pvals

results_per_filtered = results_per[(results_per['P-value'] < 1.0)]

results_protective = results_per_filtered[results_per_filtered['Subset']=='Protective']

results_deleterious = results_per_filtered[results_per_filtered['Subset']=='Deleterious']

rejected_protective, qvals_protective, _, _ = multipletests(results_protective['P-value'].values, alpha=0.10, method='fdr_bh')


rejected_deleterious, qvals_deleterious, _, _ = multipletests(results_deleterious['P-value'].values, alpha=0.10, method='fdr_bh')


results_protective['q-value'] = qvals_protective
results_protective['significant'] = rejected_protective

results_deleterious['q-value'] = qvals_deleterious
results_deleterious['significant'] = rejected_deleterious


custom_palette = {True: "#EF345C", False: "gray"}

#plot 
sns.scatterplot(results_protective, x='Odds Ratio', y='q-value', hue="significant", palette=custom_palette)
plt.axhline(correction_level)
plt.title(f"results for protective subset considering enrichment \nin amount of unique peptides with this slim (correction level = {correction_level})")

plt.xlim(np.min(results_protective['Odds Ratio'])*0.85, np.max(results_protective['Odds Ratio'])*1.15)
plt.ylim(np.min(results_deleterious['q-value'])*0.85, np.max(results_deleterious['q-value'])*1.15)


sns.despine(offset=10, trim=True)

#print all 

print(f"Enriched SLIMS in Protective peptides at Broadest Category:\n\n{results_protective[results_protective['significant']==True]['SLIM Motif'].dropna().unique()}")

plt.show()
sns.scatterplot(results_deleterious, x='Odds Ratio', y = 'q-value', hue='significant', palette=custom_palette)
plt.axhline(correction_level)

plt.xlim(np.min(results_protective['Odds Ratio'])*0.85, np.max(results_protective['Odds Ratio'])*1.15)
plt.ylim(np.min(results_deleterious['q-value'])*0.85, np.max(results_deleterious['q-value'])*1.15)

plt.title(f"Results for deleterious subset considering enrichment \nin amount of unique peptides with this slim (correction level) = {correction_level}")
sns.despine(offset=10, trim=True)
print(f"Enriched SLIMS in Deleterious Broadest Category:\n\n{results_deleterious[results_deleterious['significant']==True]['SLIM Motif'].dropna().unique()}")

plt.show()


#save



results_protective.to_csv("SLIMs/Results/Tables/Results_per_BROAD_Protective.csv",header=True,index=None)
results_deleterious.to_csv("SLIMs/Results/Tables/Results_per_BROAD_deleterious.csv",header=True,index=None)
